# Calibrate Ore-Model Erosion-Ratio Hard/Normal Metric

Calibrate the same local erosion-ratio metric as `06_calibrate_erosion_ratio_intergrowth_metric.ipynb`, but calculate the mineral mask directly with the trained multiclass ore segmentation checkpoint.

The ore outline used for scoring is derived from non-background mineral labels in the ore segmentation output. No binary segmentation checkpoint or saved `ore_mask.png` prediction artifacts are used.

The metric is calculated per local mineral-mask window as `sum(area(eroded class_i mask)) / sum(area(class_i mask))`, then bilinearly interpolated to full image size. Sparse windows with ore area below `MIN_ORE_FRACTION` receive metric `0`. High erosion ratio is treated as normal ore; low erosion ratio is treated as hard ore.

In [ ]:
from pathlib import Path
import json
import sys

from PIL import Image
import matplotlib.pyplot as plt
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ore_detection.data.inventory import inventory_baseline_images
from ore_detection.descriptors.erosion_ratio import (
    ErosionRatioConfig,
    choose_erosion_ratio_threshold,
    classify_erosion_ratio_intergrowth,
    mean_ore_ratio_score,
    save_erosion_ratio_config,
)
from ore_detection.inference.model_prediction import (
    binary_mask_from_class_image,
    load_simple_unet_checkpoint,
    predict_multiclass_segmentation_image,
)

BASELINE_ROOT = PROJECT_ROOT / 'datasets' / 'baseline'
ORE_MODEL_SERIAL = '001'
ORE_CHECKPOINT = PROJECT_ROOT / 'models' / 'source_ore_segmentation' / ORE_MODEL_SERIAL / 'best.pt'
CONFIG_PATH = PROJECT_ROOT / 'models' / 'intergrowth_erosion_ratio_ore_segmentation' / '001' / 'classifier.json'

RUN_CALIBRATION = False
SAVE_CALIBRATED_CONFIG = False
RUN_VISUAL_REVIEW = True
SAMPLE_LIMIT = None          # None means all baseline crops during calibration
REVIEW_LIMIT = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

EROSION_KERNEL_SIZE = 5      # square erosion window size; must be odd
EROSION_ITERATIONS = 3       # erosion time / repeat count
WINDOW_SIZE = 128            # local metric window size
NORMAL_THRESHOLD = 0.4       # normal if interpolated metric is greater than this
MIN_ORE_FRACTION = 0.05      # metric is 0 when ore area in the window is below 5%

config = ErosionRatioConfig(
    erosion_kernel_size=EROSION_KERNEL_SIZE,
    erosion_iterations=EROSION_ITERATIONS,
    window_size=WINDOW_SIZE,
    normal_threshold=NORMAL_THRESHOLD,
    min_ore_fraction=MIN_ORE_FRACTION,
)
config.validate()

print(f'baseline root: {BASELINE_ROOT}')
print(f'ore checkpoint: {ORE_CHECKPOINT}')
print(f'config path: {CONFIG_PATH}')
print(f'device: {DEVICE}')
print(json.dumps(config.to_dict(), indent=2))

## Load Baseline Crop Records

This notebook uses weak baseline crop labels for threshold calibration. Panorama images are skipped by `inventory_baseline_images`.

In [ ]:
baseline_records = inventory_baseline_images(BASELINE_ROOT)
baseline_records = sorted(baseline_records, key=lambda row: (row['part'], row['label'], row['path']))
label_counts = {}
for record in baseline_records:
    label_counts[str(record['label'])] = label_counts.get(str(record['label']), 0) + 1

print('baseline crop count (panoramas skipped):', len(baseline_records))
print(label_counts)


def records_with_sample_limit(records):
    return records if SAMPLE_LIMIT is None else records[:SAMPLE_LIMIT]

## Load Ore Segmentation Model

The trained multiclass ore segmentation checkpoint predicts class-index mineral masks. The binary ore mask for this notebook is derived from `class_index != background_index`.

In [ ]:
if not ORE_CHECKPOINT.exists():
    raise FileNotFoundError(f'ore segmentation checkpoint not found: {ORE_CHECKPOINT}')

ore_model = load_simple_unet_checkpoint(ORE_CHECKPOINT, device=DEVICE)
BACKGROUND_INDEX = 0 if ore_model.metadata.background_index is None else int(ore_model.metadata.background_index)

print('ore model task:', ore_model.metadata.task)
print('background index:', BACKGROUND_INDEX)
print('class names:')
for index, name in enumerate(ore_model.metadata.class_names):
    marker = ' (background)' if index == BACKGROUND_INDEX else ''
    print(f'  {index}: {name}{marker}')

## Predict Mineral Masks From Ore Model

`predict_ore_model_masks` runs the ore segmentation model for one baseline crop, returns the raw multiclass mineral mask, and derives an ore mask from non-background mineral labels. These masks are kept in memory; this notebook does not write prediction artifacts.

In [ ]:
def predict_ore_model_masks(record):
    image_path = Path(record['path'])
    with Image.open(image_path) as opened:
        prediction = predict_multiclass_segmentation_image(opened.convert('RGB'), ore_model=ore_model)
    mineral_mask = prediction.multiclass_mask
    ore_mask = binary_mask_from_class_image(mineral_mask, background_index=BACKGROUND_INDEX)
    return ore_mask, mineral_mask, prediction.multiclass_confidence


def image_score_from_ore_model(record, config):
    ore_mask, mineral_mask, _confidence = predict_ore_model_masks(record)
    result = classify_erosion_ratio_intergrowth(
        ore_mask,
        multiclass_mask=mineral_mask,
        background_index=BACKGROUND_INDEX,
        config=config,
    )
    return mean_ore_ratio_score(result.normal_score, ore_mask)

## Calibrate Threshold

The image-level calibration proxy is the mean interpolated erosion-ratio score over ore pixels derived from the ore model. Normal ore should have higher values; hard ore should have lower values. `Talc contained` crops are skipped for threshold fitting because talc remains a separate manual annotation class.

Set `RUN_CALIBRATION = True` to run this over the selected baseline crops. Set `SAVE_CALIBRATED_CONFIG = True` only when the visual results are acceptable and you want to write the separate ore-model config.

In [ ]:
labeled_scores = []
diagnostic_scores = []
processed_counts = {}

if RUN_CALIBRATION:
    calibration_records = records_with_sample_limit(baseline_records)
    print(f'calibration input crop count: {len(calibration_records)}')
    for index, record in enumerate(calibration_records, start=1):
        label = str(record['label'])
        score = image_score_from_ore_model(record, config)
        processed_counts[label] = processed_counts.get(label, 0) + 1
        if label in {'Normal ore', 'Hard ore'}:
            labeled_scores.append((score, label))
        else:
            diagnostic_scores.append((score, label, record['path']))
        if index % 25 == 0:
            print(f'processed {index}/{len(calibration_records)} crops')

    print('processed counts:', processed_counts)
    labels_with_scores = {label for _, label in labeled_scores}
    if not {'Normal ore', 'Hard ore'} <= labels_with_scores:
        print('Need both Normal ore and Hard ore crops before saving a calibrated threshold.')
        print(json.dumps(config.to_dict(), indent=2))
    else:
        threshold = choose_erosion_ratio_threshold(labeled_scores)
        calibrated = ErosionRatioConfig(
            erosion_kernel_size=EROSION_KERNEL_SIZE,
            erosion_iterations=EROSION_ITERATIONS,
            window_size=WINDOW_SIZE,
            normal_threshold=threshold['threshold'],
            min_ore_fraction=MIN_ORE_FRACTION,
        )
        config = calibrated
        print('chosen threshold:', threshold)
        if SAVE_CALIBRATED_CONFIG:
            save_erosion_ratio_config(calibrated, CONFIG_PATH)
            print('saved:', CONFIG_PATH)
        else:
            print('SAVE_CALIBRATED_CONFIG is False; config not saved.')
        print(json.dumps(config.to_dict(), indent=2))
else:
    print('RUN_CALIBRATION is False; using manual parameters only.')
    print('Set RUN_CALIBRATION = True to score baseline crops with the ore segmentation model.')
    print(json.dumps(config.to_dict(), indent=2))

## Visual Review

Each row shows raw image, ore-model mineral mask, hard ore score, and normal ore score. Scores use the same ore-model mineral mask shown in the second panel; no saved binary segmentation mask is loaded.

In [ ]:
if RUN_VISUAL_REVIEW:
    review_records = baseline_records[:REVIEW_LIMIT]
    print(f'visual review crop count: {len(review_records)}')
    for record in review_records:
        image_path = Path(record['path'])
        with Image.open(image_path) as raw:
            raw_rgb = raw.convert('RGB')

        ore_mask, mineral_mask, _confidence = predict_ore_model_masks(record)
        result = classify_erosion_ratio_intergrowth(
            ore_mask,
            multiclass_mask=mineral_mask,
            background_index=BACKGROUND_INDEX,
            config=config,
        )
        mean_score = mean_ore_ratio_score(result.normal_score, ore_mask)

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(raw_rgb)
        axes[0].set_title('raw image')
        axes[1].imshow(mineral_mask, cmap='tab20', interpolation='nearest')
        axes[1].set_title('ore-model mineral mask')
        axes[2].imshow(result.hard_score, cmap='magma', vmin=0, vmax=255)
        axes[2].set_title('hard ore score')
        axes[3].imshow(result.normal_score, cmap='viridis', vmin=0, vmax=255)
        axes[3].set_title('normal ore score')
        fig.suptitle(
            f"{record['part']} / {record['label']} / {Path(record['path']).name} | "
            f"mean erosion ratio={mean_score:.3f} | "
            f"label={result.metrics['image_label']}"
        )
        for ax in axes:
            ax.axis('off')
        plt.show()
else:
    print('RUN_VISUAL_REVIEW is False; skipping visual review.')